# Esplorare il modello

Questo notebook serve a **mettere le mani sul modello**: eseguirlo, cambiare i parametri,
guardare i risultati e verificare che le proposizioni dell'articolo reggano.
Ogni cella è indipendente dalle successive; eseguile in ordine la prima volta.

1. Un run del modello base
2. Repliche e intervalli di confidenza
3. L'esperimento di meccanismo (la *skill cliff*)
4. Verifica della Proposizione 1 (B emergente) e della Proposizione 2 (ritardo)
5. Giocare con i parametri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skillcliff import load_config
from skillcliff.ai import MeetingShock
from skillcliff.experiment import run_replications, replication_seeds
from skillcliff.learning import role_masks
from skillcliff.metrics import mean_ci
from skillcliff.model import Model, run_model

cfg = load_config("../configs/base.yaml")
cfg.meetings, cfg.roles, cfg.learning

## 1. Un run del modello base
Una replica: 120 anni di burn-in e 60 registrati (circa 0,1 s).

In [ ]:
r = run_model(cfg, replication_seeds(cfg, 1)[0])
print(f"B = {r['B'].mean():.4f}  (da incontri {r['B_meet'].mean():.4f}, autonomo {r['B_aut'].mean():.4f})")
print(f"Gini dei salari = {r['gini_w'].mean():.3f};  senior/junior = {(r['n_senior'] / r['n_junior']).mean():.2f}")

ages = np.arange(cfg.max_age)
ea = cfg.population.entry_age
for key, lab, a0 in (("h_by_age_low", "non qualificati", ea.low), ("h_by_age_high", "qualificati", ea.high)):
    plt.plot(ages[a0:], r[key][:, a0:].mean(axis=0), label=lab)
plt.xlabel("età"); plt.ylabel("salario medio (w = h)"); plt.legend(); plt.title("Profilo salariale per età");

## 2. Repliche e intervalli di confidenza
30 repliche con seed derivati (circa 2 s). Stesso seed ⇒ stessi risultati.

In [ ]:
res = run_replications(cfg)
for k in ("B", "B_meet", "mean_h_junior", "mean_h_senior", "gini_w"):
    m, lo, hi = mean_ci(res[k].mean(axis=1))
    print(f"{k:14s} {m:.4f}   IC95 [{lo:.4f}, {hi:.4f}]")

## 3. L'esperimento di meccanismo
Dall'anno 10 la probabilità d'incontro `p` viene dimezzata. Non è ancora l'IA: isola il canale degli incontri.

In [ ]:
shock = run_replications(cfg, ai_factory=lambda c: MeetingShock(c, year=10, p_mult=0.5))
years = np.arange(cfg.T)
for k, lab in (("mean_h_junior", "junior"), ("mean_h_senior", "senior")):
    rel = shock[k] / res[k] - 1
    m, lo, hi = mean_ci(rel)
    plt.fill_between(years, lo, hi, alpha=.2); plt.plot(years, m, label=lab)
plt.axvline(10, ls=":", c="gray"); plt.axhline(0, c="gray", lw=.5)
plt.xlabel("anno"); plt.ylabel("variazione rispetto al base"); plt.legend(); plt.title("La skill cliff");

## 4. Verifica delle proposizioni
**Proposizione 2 (ritardo).** L'h dei senior non cambia per `s_S - s_J` anni dopo lo shock e cambia dall'anno successivo.

In [ ]:
gap = (shock["mean_h_senior"] / res["mean_h_senior"] - 1).mean(axis=0)
lag = cfg.roles.senior_min_exp - cfg.roles.junior_max_exp + 1
print("ritardo previsto:", lag, "anni")
for k in range(lag - 2, lag + 3):
    print(f"  anno shock + {k:2d}: {gap[10 + k]:+.5f}")

**Proposizione 1 (B emergente).** Con incontri casuali dentro ciascun gruppo di qualifica $g$:

$E[B^{inc}] = \sum_g \frac{J_g}{J}\, \beta\, p^{eff}_g \left[\bar h^S_g\, \overline{h^{-1}}^J_g - 1 + \Omega_g\right]$, con $\Omega_g$ il termine di troncamento (i junior non disimparano).

In [ ]:
m = Model(cfg, replication_seeds(cfg, 1)[0])
for _ in range(cfg.burn_in):
    m.step()
sim, formula = [], []
for _ in range(30):
    rec = m.step()
    pop = m.pop
    h0 = pop.h - m.last_meet.dh - m.last_dh_aut          # h a inizio anno
    junior, senior = role_masks(cfg, pop.exp)
    groups = (~pop.high, pop.high) if cfg.meetings.same_qual else (np.ones(len(pop), bool),)
    f = 0.0
    for g in groups:
        hJ, hS = h0[junior & g], h0[senior & g]
        p_g = min(cfg.meetings.p, cfg.meetings.kappa * hS.size / hJ.size)
        omega = np.mean(np.maximum(0, hJ[:, None] - hS[None, :]) / hJ[:, None])
        f += hJ.size / junior.sum() * cfg.meetings.beta * p_g * (hS.mean() * np.mean(1 / hJ) - 1 + omega)
    sim.append(rec["B_meet"])
    formula.append(f)
print(f"B da incontri simulato: {np.mean(sim):.5f}   identità: {np.mean(formula):.5f}")

## 5. Giocare con i parametri
Cambia i valori e riesegui. `with_overrides` usa la notazione puntata dei parametri in `configs/base.yaml`.

In [ ]:
prova = cfg.with_overrides({
    "meetings.p": 0.6,
    "meetings.kappa": 0.2,
    "meetings.beta": 0.10,
    "meetings.same_qual": True,
    "roles.junior_max_exp": 5,
    "roles.senior_min_exp": 15,
})
r2 = run_replications(prova, n_reps=10)
print(f"B = {r2['B'].mean():.4f};  h senior = {r2['mean_h_senior'].mean():.3f};  p_eff = {r2['p_eff'].mean():.2f}")